# AGENTIC AI ASSISTANT — screen the latest lead list against the lending policy

**What this notebook does:** connects the two halves of the project.

```
lead-scoring pipeline                          RM Copilot agent (genai/copilot)
client/output/<latest>/leads.csv  ──►  for each company:  research → signal → policy → supervisor → brief
   100 ranked prospects                                                                    │
                                                        RM lead list + one cited brief per lead + charts
```

The scoring model ranks the **population**: who looks likely to need secured lending. The agent screens the
**individuals**. It pulls each company's Companies House record, applies the 23-clause lending policy, and
writes a brief the relationship manager (RM) can act on, citing the rule behind every decision.

**What it deliberately does not do:** re-score or re-rank. The lead list's order is kept; the agent adds a
policy decision next to each lead and removes the ones policy rules out. Decisions come from code (the
16 clause rules). Two models do the rest (notebooks 02–04):

- **Claude Opus 5** reads free text that can feed a verdict: charge particulars, and pre-2013 charges;
- **Claude Sonnet 5** writes the brief's summary. That is writing from a fact sheet the code has already
  checked, and it never sets a decision, so the cheaper model does it (2.5× cheaper per token).

**Cost and time, for 100 leads:** Companies House calls are paced to the API limit (~2 per second). Every
lead is **screened**: its decision is made in code, and the brief's decision, conditions and clause table are
written by code, so screening costs almost nothing. Only the top `BRIEF_TOP_N` leads that pass screening also get a
**model-written summary**, which is where nearly all the cost is. The last section reports the actual figures.

Prerequisites: `.env` has `CH_API` and `ANTHROPIC_API_KEY`; kernel is *Python 3.13 (Lloyds .venv)*.

## 1 · Setup

The whole graph now lives in the `copilot/` package (research, signal, policy, supervisor, brief), so this
notebook only orchestrates: find the list, fill the cache, run the graph per company, and read the results.

Results are written to `genai/agentic_output/<lead-list date>/`, one JSON state per company. **A re-run
resumes:** companies already assessed under the *same policy version* are loaded from disk, not re-run and
re-paid for. A company assessed under an older policy version is re-assessed automatically, so one run never
mixes two versions. Set `FORCE = True` to redo every company.

In [ ]:
import asyncio, json, re, sys, time
from datetime import date
from pathlib import Path

import matplotlib as mpl
import matplotlib.patches
import matplotlib.pyplot as plt
import matplotlib.ticker
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

ROOT  = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists())
GENAI = ROOT / "genai"
load_dotenv(ROOT / ".env")
sys.path.insert(0, str(GENAI))

from copilot import llm
from copilot.graph import build
from copilot.policy import CLAUSE, CONDITION, decide, level
from copilot.research import load_tools, open_tools
from copilot import batch                        # screen + summarise, shared with the web app (webapp/)

CONCURRENCY = 4          # companies assessed at once (all sharing one MCP server process: research.open_tools)
BRIEF_TOP_N = 20         # model-written summaries for the N highest-ranked PROCEED/REFER leads; every lead is screened
FORCE       = False      # True = re-assess companies that already have a saved state
llm.VERBOSE = False      # per-call token lines off; totals are reported in section 9
print("repo root:", ROOT)
print(f"models   : {llm.MODEL} reads · {llm.BRIEF_MODEL} writes summaries (top {BRIEF_TOP_N} prospects)")

## 2 · Find the latest lead list

The scoring pipeline (`5_score.ipynb`) writes one folder per run under `client/output/`. The latest date is
the current list. Its `run_manifest.txt` records how the list was made, including how many existing
customers it excluded.

In [ ]:
RUNS = sorted(p for p in (ROOT / "client" / "output").iterdir() if p.is_dir() and p.name[:2] == "20")
LEAD_DIR = RUNS[-1]
leads = pd.read_csv(LEAD_DIR / "leads.csv", dtype={"com_num": str}).sort_values("rank").reset_index(drop=True)
manifest = (LEAD_DIR / "run_manifest.txt").read_text()

OUT = GENAI / "agentic_output" / LEAD_DIR.name
for d in ("states", "briefs", "charts"):
    (OUT / d).mkdir(parents=True, exist_ok=True)

# the assessment date is fixed on the first run and reused, so a resumed run stays consistent
info_path = OUT / "run_info.json"
RUN = json.loads(info_path.read_text()) if info_path.exists() else {"as_of": date.today().isoformat()}
POLICY = batch.policy_versions()
info_path.write_text(json.dumps({**RUN, "lead_list": str(LEAD_DIR.relative_to(ROOT)), "policy": POLICY}, indent=1))
AS_OF = RUN["as_of"]

print(f"lead list : {LEAD_DIR.relative_to(ROOT)}  ({len(leads)} companies)")
print(f"assessed  : as of {AS_OF} (records as Companies House holds them on this date)")
print(f"policy    : {', '.join(f'{d} {v}' for d, v in POLICY.items())}")
print(f"output    : {OUT.relative_to(ROOT)}\n")
print(manifest)
leads[["rank", "com_num", "name", "sector", "region", "account_type", "gated"]].head(10)

**Two things in the manifest matter for reading the results.**

- `client files: sample copy.csv` means this run scored a **100-company test sample**, built in
  `01_sample.ipynb` from the training table: 25 firms that went on to take a Lloyds charge, and 75 that did not.
  It isn't a live weekly list, so the proportions below describe this sample, not a deployment.
- `excluded (cust): 0`. `5_score.ipynb` removes existing customers only when the client file marks them
  `relationship = customer`, and `01_sample.ipynb` marked all 100 as `prospect`. The agent doesn't rely on
  that column. It checks the charge register itself (section 5).

## 3 · Fill the Companies House cache — at the API's rate limit

Each research pass calls the four tools **at the same time, in four separate server processes**, and each
process paces itself independently. For one company that's fine; for 100 it could send ~7 requests per
second against a limit of 600 per 5 minutes (2 per second).

So the records are fetched first, **one call at a time in this process** (`batch.prefetch`), through the
same `mcp_ch/tools.py` functions the MCP server runs. `ch_client` spaces calls 0.55 s apart and writes every response to
`mcp_ch/cache/`. When the graph runs next, every MCP call is answered from that cache, with zero API calls.

In [ ]:
CACHE = GENAI / "mcp_ch" / "cache"
before = len(list(CACHE.glob("*.json")))
fetched = batch.prefetch(leads["com_num"].tolist(),
                         on_progress=lambda s, i, n: print(f"  {i}/{n} companies") if i % 25 == 0 else None)
print(f"new API responses cached: {len(list(CACHE.glob('*.json'))) - before} · {fetched['seconds']} s · "
      f"failures: {len(fetched['failed'])}")
for k, v in fetched["failed"].items():
    print("  ", k, v)

## 4 · The graph

Built from the package. Since notebook 04, the supervisor has one new route, checked first: **a company
holding a live Lloyds-group charge ends the run without a brief.** That is the scoring pipeline's own rule
(`3_Flat_table.ipynb`): a firm with any Lloyds charge still outstanding is an existing customer and is never a
prospect. A firm whose Lloyds charges are all satisfied is a lapsed customer, and is still assessed.

In [ ]:
graph = build(await load_tools())       # for the drawing; the run below opens one shared server (open_tools)
print(graph.get_graph().draw_ascii())

## 5 · Run the agent: screen every lead, summarise the top ones

The run is split by cost. In the first full run, the model-written summary was **94% of the spend** ($2.91 of
$3.11), while the decision itself is made in code. So there are two passes:

1. **Screen all leads** (`draft_brief=False`). Every clause is decided and the brief is written by code alone:
   decision, conditions, clause table and gaps. The model is called only to read a pre-2013 charge the rules
   can't settle. Collateral isn't classified, because it feeds only the summary.
2. **Summarise the top `BRIEF_TOP_N` leads that passed screening** (decision PROCEED or REFER, not an existing
   customer; `draft_brief=True`), in the lead list's order: the full graph,
   with the model-written summary. This is what an RM reads first; the rest of the list keeps its decision.

For each run:

- **LangSmith** gets ONE trace for the whole batch (`rm-copilot run · <list>`). Every company's graph run
  (`rm-copilot <company number>`) nests inside it, and every Claude call inside those with its tokens and
  cost, so the trace's total is the batch's cost. The cell prints its link.
- **Token usage** is recorded per company (`llm.CURRENT` tags every Claude call with the company being assessed).
- The **final state** is saved to `states/<company>.json`, the audit record: every clause's verdict, the
  supervisor log, the brief, and the statements dropped. The brief also goes to `briefs/` as Markdown.

Both passes are `batch.screen_and_summarise` in `copilot/batch.py`, the same code the web app runs. A saved
state is reused when its policy version matches and it covers what's asked for. A full run also covers
screening, and a summary is reused only if the current summary model wrote it. Failed runs are always retried.

In [ ]:
t0 = time.time()
CONTEXT = {l.com_num: {"rank": int(l.rank), "of": len(leads), "score": round(float(l.score), 3),
                       "why": l.why if isinstance(getattr(l, "why", None), str) else ""}
           for l in leads.itertuples()}          # the scoring model's view, for each brief's "why this lead"
RUN_INFO = {}
# One MCP server process for the whole batch. load_tools() would start a new one for every tool call:
# ~1 s each, which was nearly all of research's time (batch trace, 30 Sep) — 16 ms per company with this.
async with open_tools() as TOOLS:
    STATES, TOP = await batch.screen_and_summarise(
        build(TOOLS), list(zip(leads["com_num"], leads["rank"].astype(int))), OUT, AS_OF, BRIEF_TOP_N,
        concurrency=CONCURRENCY, force=FORCE, lead_list=LEAD_DIR.name, context=CONTEXT, info=RUN_INFO)
STATES = {n: STATES[n] for n in leads["com_num"]}             # rank order

rank = dict(zip(leads["com_num"], leads["rank"]))
errors = pd.Series([s["error"] for s in STATES.values() if "error" in s], dtype=str)
span = f" (ranks {rank[TOP[0]]}–{rank[TOP[-1]]})" if TOP else ""
print(f"{len(STATES)} companies screened · {len(TOP)} summarised{span} · "
      f"{time.time() - t0:.0f} s wall clock this session · errors: {len(errors)}")
for e, n in errors.str[:200].value_counts().items():       # the distinct reasons, so a shared cause is obvious
    print(f"  {n} × {e}")
print(f"this session: {RUN_INFO.get('model_calls')} model calls, ${RUN_INFO.get('cost_usd') or 0:.2f}")
if RUN_INFO.get("trace_url"):
    print("the whole batch in LangSmith (one trace):", RUN_INFO["trace_url"])

## 6 · One row per company

The state of each run, flattened. `outcome` is what the charts use:

- **Existing customer** when a live Lloyds-group charge ended the run;
- otherwise the policy decision under EVD-07, with **", gaps open"** added when some clause couldn't be checked
  and the missing answer could still make the decision stricter (`qualifies` is None).

`conditions` lists what must be obtained before an offer (PROCEED WITH CONDITION clauses, such as CON-04).
It never changes the decision.

In [ ]:
labels = pd.read_csv(GENAI / "final_sample.csv", dtype={"com_num": str}).set_index("com_num")["label"]

def call_cost(u: dict) -> float:
    price = next((v for k, v in llm.PRICE_PER_MTOK.items() if u["model"].startswith(k)), None)
    return (u["input_tokens"] * price[0] + u["output_tokens"] * price[1]) / 1e6 if price else float("nan")


def row(lead, st: dict) -> dict:
    s = st.get("signals") or {}
    ch = s.get("charges") or {}
    live_own = sorted(set(ch.get("own_group", [])) & set(ch.get("live", [])))
    outcome = st.get("outcome") or {}
    usage = st["_run"]["usage"]
    gaps = st.get("evidence_gap", [])
    return {
        "rank": int(lead.rank), "com_num": lead.com_num, "name": lead.name, "sector": lead.sector,
        "region": lead.region, "lead_score": round(lead.gated, 3),
        "company_status": (st.get("profile") or {}).get("company_status"),
        "error": st.get("error"),
        "existing_customer": bool(live_own),
        "lapsed_customer": bool(ch.get("own_group")) and not live_own,
        "decision": outcome.get("decision"),
        "decided_by": ", ".join(outcome.get("clauses", [])),
        "conditions": ", ".join(outcome.get("conditions", [])),
        "qualifies": st.get("qualifies"),
        "applicable": ", ".join(a["clause_id"] for a in st.get("applicable", [])),
        "gap_clauses": ", ".join(sorted({g.split(" ")[0] for g in gaps})),
        "n_gaps": len(gaps),
        "research_passes": st.get("research_attempts", 0) + 1,
        "brief": bool(st.get("brief")),
        "summary": bool(st.get("brief")) and st["_run"].get("draft_brief", True),   # model-written, top N only
        "statements_dropped": len(st.get("brief_dropped", [])),
        "model_calls": len(usage),
        "cost_usd": sum(call_cost(u) for u in usage),
        "input_tokens": sum(u["input_tokens"] for u in usage),
        "output_tokens": sum(u["output_tokens"] for u in usage),
        "seconds": st["_run"]["seconds"],
        "converted_2021_25": int(labels.get(lead.com_num, -1)),   # evaluation only; the agent never sees it
    }

res = pd.DataFrame([row(l, STATES[l.com_num]) for l in leads.itertuples()])
res["outcome"] = (res["decision"] + res["qualifies"].isna().map({True: ", gaps open", False: ""})
                  ).where(~res["existing_customer"], "Existing customer")
res.loc[res["error"].notna(), "outcome"] = "Error"
res.to_csv(OUT / "leads_screened.csv", index=False)
print(res["outcome"].value_counts().to_string())
res.head(10)

## 7 · Charts

Colours carry the outcome and are the same in every chart. PROCEED, REFER and DECLINE use the fixed status
colours (good, warning, critical). INSUFFICIENT EVIDENCE is violet, because "can't decide yet" is not a
severity level. Existing customers are grey: removed, not judged. **Hatching** marks a decision with gaps
still open. Every colour also has a text label, and the numbers are in the table in section 6.

In [ ]:
SURFACE, INK, INK2, MUTED, GRID, BASE = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
ORDER = ["Existing customer", "DECLINE", "INSUFFICIENT EVIDENCE", "REFER", "REFER, gaps open",
         "PROCEED, gaps open", "PROCEED"]
COLOR = {"Existing customer": "#898781", "DECLINE": "#d03b3b", "INSUFFICIENT EVIDENCE": "#4a3aa7",
         "REFER": "#fab219", "PROCEED": "#0ca30c", "Error": "#52514e"}
LIGHT_FILL = {"REFER", "PROCEED", "Existing customer"}         # dark ink on these, white on the rest
base  = lambda k: k.split(",")[0]                               # "REFER, gaps open" -> "REFER"
hatch = lambda k: "///" if "gaps open" in k else None           # texture = decided, but gaps still open
ink   = lambda k: INK if base(k) in LIGHT_FILL else "#ffffff"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": ["Helvetica Neue", "Arial", "DejaVu Sans"], "font.size": 10.5,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": INK2,
    "axes.edgecolor": BASE, "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "x", "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
})

def titled(ax, title, sub):
    ax.set_title(title, loc="left", fontsize=13, fontweight="bold", color=INK, pad=26)
    ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=10, color=INK2, va="bottom")

def save(fig, name):
    fig.savefig(OUT / "charts" / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()

### 7a · Where the 100 scored leads end up

In [ ]:
counts = res["outcome"].value_counts().reindex(ORDER + (["Error"] if (res["outcome"] == "Error").any() else []),
                                               fill_value=0)
counts = counts[(counts > 0) | ~counts.index.str.contains("gaps open")]     # the five decisions always show
fig, ax = plt.subplots(figsize=(9, 0.5 * len(counts) + 1.2))
y = range(len(counts))[::-1]
for yy, k, v in zip(y, counts.index, counts.values):
    ax.barh(yy, v, color=COLOR[base(k)], hatch=hatch(k), height=0.62, edgecolor=SURFACE, linewidth=2)
ax.set_yticks(list(y), counts.index)
for yy, v in zip(y, counts.values):
    ax.text(v + 0.6, yy, f"{v}", va="center", fontsize=10.5, color=INK)
ax.set_xlim(0, max(counts.max() * 1.15, 5)); ax.set_xlabel("companies")
titled(ax, f"Where the {len(res)} scored leads end up",
       f"Lead list {LEAD_DIR.name} · policy outcome per company, assessed {AS_OF}")
save(fig, "7a_outcomes")

**Of the 100 leads, 20 are already Lloyds customers. Of the 80 prospects, 1 is DECLINED and 79 are REFER.
None is PROCEED, and none is INSUFFICIENT EVIDENCE.**

- **20 existing customers.** Each holds a live charge in favour of a Lloyds-group lender. The supervisor ended the
  run before the brief, so each cost about $0.004. The scoring pipeline let them through because it trusts the
  client file's `relationship` column (section 2).
- **1 DECLINE, under SEC-05 (recently banked elsewhere).** McCulloch Enterprises 2 Ltd (rank 21) registered a
  charge to another lender on 2026-07-08, 84 days before assessment. Under the policy, its funding need is presumed
  met.
- **0 INSUFFICIENT EVIDENCE.** Every company's four records were fetched, so EVD-01 never fired. Nine prospects
  do have gaps (7d), but every gap is on a REFER clause and all nine are REFER already. A gap can add conditions
  to the brief, but it can't change the decision.
- **0 PROCEED.** This is the main finding of the run; 7c and 7e explain it.

### 7b · Policy outcome by lead rank

In [ ]:
res["band"] = pd.cut(res["rank"], bins=range(0, len(res) + 20, 20),
                     labels=[f"ranks {i+1}–{i+20}" for i in range(0, len(res), 20)])
tab = pd.crosstab(res["band"], res["outcome"]).reindex(columns=[c for c in ORDER if c in res["outcome"].unique()],
                                                        fill_value=0)
share = tab.div(tab.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(9.5, 3.8))
y = list(range(len(share)))[::-1]
left = [0.0] * len(share)
for col in share.columns:
    vals = share[col].values
    ax.barh(y, vals, left=left, color=COLOR[base(col)], hatch=hatch(col), height=0.66, edgecolor=SURFACE,
            linewidth=2, label=col)
    for yy, l, v, n in zip(y, left, vals, tab[col].values):
        if v >= 0.08:
            ax.text(l + v / 2, yy, str(n), ha="center", va="center", fontsize=10, color=ink(col),
                    bbox=dict(boxstyle="round,pad=0.15", fc=COLOR[base(col)], ec="none") if hatch(col) else None)
    left = [a + b for a, b in zip(left, vals)]
ax.set_yticks(y, share.index.astype(str)); ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.legend(ncol=min(len(share.columns), 5), loc="upper left", bbox_to_anchor=(0, -0.13), frameon=False, fontsize=9.5)
titled(ax, "Policy outcome by lead rank", "Share of each 20-company band · numbers are company counts")
save(fig, "7b_outcome_by_rank")

**Existing customers sit at the top of the list.** 11 of the 20 highest-ranked leads already bank with Lloyds,
against 3, 3, 2 and 1 in the lower bands. This is what you'd expect: the model learned what Lloyds borrowers look
like, and these companies *are* Lloyds borrowers. Without the agent's check, more than half of an RM's first 20
calls would go to the bank's own customers. (The sample was built with 25 converters, so a live week would have
fewer.)

**Among prospects, the policy outcome doesn't change with rank.** REFER dominates every band. The two halves
answer different questions. The score says *who is likely to borrow*; the policy says *whether, and on what
terms, Lloyds can lend*. A high score doesn't make a company easier to lend to, which is why the agent keeps the
model's order and doesn't re-rank.

### 7c · Which clauses fire among the prospects

In [ ]:
prospects = res[res["decision"].notna() & ~res["existing_customer"]]
hits = (prospects["applicable"].str.split(", ").explode().loc[lambda s: s.str.len() > 0].value_counts())
lab = [f"{c}  {CLAUSE[c]['title'].split('—', 1)[1].strip()}" + ("  (condition)" if c in CONDITION else "")
       for c in hits.index]
fig, ax = plt.subplots(figsize=(9.5, 0.42 * len(hits) + 1.4))
y = list(range(len(hits)))[::-1]
ax.barh(y, hits.values, color=[COLOR[level(CLAUSE[c]["outcome"])] for c in hits.index],
        height=0.66, edgecolor=SURFACE, linewidth=2)
ax.set_yticks(y, lab)
for yy, v in zip(y, hits.values):
    ax.text(v + 0.3, yy, str(v), va="center", fontsize=10, color=INK)
ax.set_xlabel(f"prospects the clause applies to (of {len(prospects)})")
handles = [mpl.patches.Patch(color=COLOR[k], label=f"clause outcome: {k}"
                             + (" (incl. WITH CONDITION)" if k == "PROCEED" and set(hits.index) & set(CONDITION) else ""))
           for k in ["DECLINE", "INSUFFICIENT EVIDENCE", "REFER", "PROCEED"]
           if any(level(CLAUSE[c]["outcome"]) == k for c in hits.index)]
ax.legend(handles=handles, loc="lower right", frameon=False, fontsize=9)
titled(ax, "Which policy clauses apply", f"{len(prospects)} prospects assessed (existing customers excluded)")
save(fig, "7c_clauses")

**CON-04 applies to 77 of the 80 prospects.** It refers any company whose latest accounts are micro-entity or
total-exemption, because those accounts don't show turnover or profit. But those account types are most of what
the project's SME filter lets in, so CON-04 fires on almost every lead the model can produce. For 33 prospects
it's the *only* REFER clause. Section 7e shows what that costs.

**The security side is mostly clean.** SEC-02 (no live charges) applies to 67 prospects. Where a prospect isn't
clean, the reasons are usually about filing conduct: a pattern of late filing (CON-03, 23 prospects) or accounts
filed more than 30 days late (CON-01, 20).

**SEC-07 (negative pledge) applies to 14 prospects. For 5 of them, every charge behind it is already fully
satisfied.** For those 5, SEC-03 says the same charges are to be disregarded. The policy contradicts itself on
the same company. This is the open question the `xfail` test tracks, now with real cases.

**A bug this run exposed, now fixed.** The first run DECLINED Aarvi Encon Resources Ltd (rank 91) under CON-07,
insolvency proceedings. The evidence was a *solvency* statement (form CAP-SS): directors file it when they reduce
share capital, and it declares the company solvent. Companies House puts it in its "insolvency" category, and the
rule read that whole category as insolvency. The fixes:

- `copilot/signals.py` now excludes CAP-SS;
- `tests/test_rules.py` has a regression test;
- the company was re-assessed: it is REFER under CON-04.

### 7d · What the agent couldn't decide, and why

In [ ]:
def cause(gap: str) -> str:
    if "window truncated" in gap:
        return "filing history cut at 60 items (CON-01/03/05)"
    if "record missing" in gap or gap.startswith("EVD-01"):
        return "a record is missing (EVD-01)"
    if "quote not found" in gap:
        return "model answer failed the quote check"
    if gap.startswith(("SEC-07", "SEC-08")):
        return "pre-2013 charge: detail only in an image (SEC-07/08)"
    return "other"

gap_rows = [(r.com_num, cause(g)) for r in prospects.itertuples()
            for g in STATES[r.com_num].get("evidence_gap", [])]
by_cause = pd.DataFrame(gap_rows, columns=["com_num", "cause"]).drop_duplicates().value_counts("cause")
fig, ax = plt.subplots(figsize=(9, 0.5 * max(len(by_cause), 1) + 1.4))
y = list(range(len(by_cause)))[::-1]
ax.barh(y, by_cause.values, color="#2a78d6", height=0.6, edgecolor=SURFACE, linewidth=2)
ax.set_yticks(y, by_cause.index)
for yy, v in zip(y, by_cause.values):
    ax.text(v + 0.2, yy, str(v), va="center", fontsize=10, color=INK)
ax.set_xlabel("prospects with at least one gap of this kind")
titled(ax, "Where the evidence ran out",
       f"{int((prospects['n_gaps'] > 0).sum())} of {len(prospects)} prospects have at least one clause the record can't settle")
save(fig, "7d_gaps")

**9 of the 80 prospects have at least one clause the record can't settle (one of them has both kinds of gap).
None of these gaps changes a decision.**

- **8 prospects: filing history cut at 60 items.** These are long-established companies with many filings. The
  tool keeps the latest 60, so for CON-01 and CON-03 the agent can't rule out a late filing earlier in the three
  years. It records a gap instead of assuming there was none.
- **2 prospects: a pre-2013 charge whose detail is only in a scanned image.** Charges registered before April 2013
  carry no flag for a negative pledge or a floating charge. The model was asked to read the text, found only "see
  image", and said so rather than guess.

Every gap is on a REFER clause, and all 9 companies are REFER already. That's why INSUFFICIENT EVIDENCE is 0 in
7a. Each of the 9 briefs carries the thin-evidence heading and lists what is missing.

### 7e · What the CON-04 change did, and what SEC-07 would do

**Policy v1.1 changed CON-04** (earnings not disclosed) from REFER to PROCEED WITH CONDITION: the RM still
has to obtain 12 months of management accounts, but that no longer decides the outcome. Screening this list
under v1.0 showed CON-04 applied to almost every SME prospect.

This chart re-applies EVD-07 to the saved verdicts under each version. It makes **no new calls**: the clause
verdicts are the same, and only how they combine changes. The third bar is the other open question: **SEC-07**
(negative pledge) also counts charges that are fully satisfied, while SEC-03 says to disregard those. That
change is not made; it's the policy owner's call, tracked by the `xfail` test.

In [ ]:
def what_if(st: dict, refer=(), sec07_live_only=False) -> str:
    live = set(((st.get("signals") or {}).get("charges") or {}).get("live", []))
    kept = [{**a, "outcome": "REFER"} if a["clause_id"] in refer else a            # v1.0: the clause referred
            for a in st.get("applicable", [])
            if not (sec07_live_only and a["clause_id"] == "SEC-07"
                    and not any(e.split(": «")[0] in live for e in a["evidence"]))]
    outcome, qualifies = decide(kept, st.get("evidence_gap", []))
    return outcome["decision"] + (", gaps open" if qualifies is None else "")

SCENARIOS = {
    "v1.0: CON-04 refers":                    {"refer": ("CON-04",)},
    "v1.1: CON-04 is a condition (now)":      {},
    "v1.1 + SEC-07 on live charges only":     {"sec07_live_only": True},
}
SEGMENTS = ["DECLINE", "REFER", "REFER, gaps open", "PROCEED, gaps open", "PROCEED"]
wi = pd.DataFrame({name: pd.Series([what_if(STATES[n], **kw) for n in prospects["com_num"]]).value_counts()
                   for name, kw in SCENARIOS.items()}).T.reindex(columns=SEGMENTS, fill_value=0).fillna(0).astype(int)
print(wi.to_string(), "\n")

fig, ax = plt.subplots(figsize=(9.5, 3.0))
y = list(range(len(wi)))[::-1]
left = [0] * len(wi)
for seg in SEGMENTS:
    vals = wi[seg].values
    ax.barh(y, vals, left=left, color=COLOR[base(seg)], height=0.64, edgecolor=SURFACE, linewidth=2,
            hatch=hatch(seg), label=seg)
    for yy, l, v in zip(y, left, vals):
        if v >= 3:
            ax.text(l + v / 2, yy, str(v), ha="center", va="center", fontsize=10, color=ink(seg),
                    bbox=dict(boxstyle="round,pad=0.15", fc=COLOR[base(seg)], ec="none") if hatch(seg) else None)
    left = [a + b for a, b in zip(left, vals)]
ax.set_yticks(y, wi.index); ax.set_xlim(0, len(prospects)); ax.set_xlabel(f"prospects (of {len(prospects)})")
ax.legend(ncol=len(SEGMENTS), loc="upper left", bbox_to_anchor=(0, -0.24), frameon=False, fontsize=9,
          handlelength=1.6, columnspacing=1.2)
titled(ax, "The same 80 prospects under each policy version",
       "Same verdicts, EVD-07 re-applied · hatched = decided, but some clauses couldn't be checked")
save(fig, "7e_policy_versions")

**Two clauses decide most of the outcome.**

- **CON-04 as a document request instead of a REFER.** 33 of the 80 prospects become PROCEED: 29 outright, plus 4
  with gaps still open. 41 stay REFER for other reasons. The condition itself doesn't go away: the RM still has to
  obtain 12 months of management accounts. It just stops being a reason to refer.
- **SEC-07 on live charges only.** One prospect moves: ECI 9 GP Ltd, rank 3, the top of the RM list. Its only
  REFER reason is a negative pledge on two charges that are both fully satisfied. Its brief (section 9) deals with
  this sensibly: it asks for evidence that the pledge fell away when the charges were repaid.
- **Both changes:** 36 PROCEED (32, plus 4 with gaps open).

**What this means.** As written, the policy refers nearly every SME, so REFER no longer separates one lead from
another. That isn't an agent error: the agent applied each clause exactly as written. It's a calibration
question for the policy owner, and this chart gives them the number. It only came to light by running the policy
over a real list.

*Hatched bars* mean decided with gaps open. Every clause the record could check gives that decision, but a REFER
clause couldn't be checked. So a hatched PROCEED holds only if the gap clears.

## 8 · The RM lead list

What an RM would actually receive: the scored leads **minus existing customers and DECLINEs**, in the
scoring model's order. Each lead has:

- its policy decision;
- `refer_reasons`: the REFER clauses a person must judge;
- `before_offer`: documents to obtain before an offer (the conditions);
- `gap_clauses`: any clauses the record couldn't settle;
- `brief_file`: its brief. Every lead has one; `summary` marks the top `BRIEF_TOP_N`, whose brief also has
  the model-written summary.

In [ ]:
rm = res[res["decision"].isin(["PROCEED", "REFER", "INSUFFICIENT EVIDENCE"]) & ~res["existing_customer"]].copy()
rm["refer_reasons"] = rm["applicable"].apply(
    lambda s: ", ".join(c for c in s.split(", ") if c and level(CLAUSE[c]["outcome"]) == "REFER"))
rm["before_offer"] = rm["conditions"]
rm["brief_file"] = rm.apply(lambda r: f"briefs/{r['rank']:03d}_{r['com_num']}.md" if r["brief"] else "", axis=1)
cols = ["rank", "com_num", "name", "sector", "region", "lead_score", "decision", "refer_reasons", "before_offer",
        "gap_clauses", "summary", "brief_file"]
rm[cols].to_csv(OUT / "rm_lead_list.csv", index=False)
print(f"{len(rm)} of {len(res)} leads reach the RM list · saved {(OUT / 'rm_lead_list.csv').relative_to(ROOT)}")
print(rm["outcome"].value_counts().reindex([o for o in ORDER if o in set(rm["outcome"])]).to_string(), "\n")
rm[cols].head(15)

## 9 · One brief, as the RM sees it

The highest-ranked **PROCEED** lead with a model-written summary (the top summarised lead if there is none). The line
**Before an offer** is the CON-04 condition: written by code from the clause, so it is always exact.

In [ ]:
pick = rm[rm["summary"] & (rm["outcome"] == "PROCEED")]
top = (pick if len(pick) else rm[rm["summary"]] if rm["summary"].any() else rm).iloc[0]
print("supervisor_log:", *STATES[top.com_num]["supervisor_log"], sep="\n  ")
display(Markdown(STATES[top.com_num]["brief"]))

## 10 · A check against what actually happened

This sample was built from the training table, so each company has a label: did it take a Lloyds charge
between 2021 and 2025? The agent never sees this column. Comparing it with the agent's existing-customer
check is a clean test of that check, because both come from the same charge register.

In [ ]:
print(pd.crosstab(res["converted_2021_25"].map({1: "took a Lloyds charge 2021–25", 0: "did not"}),
                  res["existing_customer"].map({True: "agent: existing customer", False: "agent: prospect"}),
                  margins=True).to_string())

**The existing-customer check agrees with the label on all 100 companies.**

- All 20 companies the agent flagged took a Lloyds charge in 2021–25, and none of the 75 non-converters was
  flagged.
- The other 5 converters are *lapsed* customers: every Lloyds charge they took is now fully satisfied. The
  pipeline's own rule (`3_Flat_table.ipynb`) treats a lapsed customer as a prospect again, so the agent assessed
  them. All 5 are REFER, with briefs.

The check and the label both come from the charge register, so agreement tests the *implementation* (lender-name
matching, live versus satisfied status) rather than adding a new source of truth. It also says nothing about the
policy decisions. The label records whether a company *borrowed from Lloyds*, not whether lending to it was within
policy. Checking those decisions needs a person to review a sample of briefs.

## 11 · Cost, time and observability

Each call is priced by the model that answered it, at Anthropic's list prices (`llm.PRICE_PER_MTOK`):
`claude-opus-5` $5 / $25 and `claude-sonnet-5` $2 / $10 per million input / output tokens. A fallback to a
model not in that table shows as NaN rather than being guessed.

In [ ]:
calls = pd.DataFrame([{**u, "cost_usd": call_cost(u)} for st in STATES.values() for u in st["_run"]["usage"]])
models = sorted(calls["model"].unique()) if len(calls) else []
print(calls.groupby(["node", "model"]).agg(calls=("cost_usd", "size"), cost_usd=("cost_usd", "sum"))
      .round(3).to_string() if len(calls) else "no model calls", "\n")
summary = pd.DataFrame({
    "companies": [len(res)],
    "model calls": [int(res["model_calls"].sum())],
    "input tokens": [int(res["input_tokens"].sum())],
    "output tokens": [int(res["output_tokens"].sum())],
    "est. cost (USD)": [round(res["cost_usd"].sum(), 2)],
    "median s / company": [res["seconds"].median()],
    "briefs (code-written parts)": [int(res["brief"].sum())],
    "model-written summaries": [int(res["summary"].sum())],
    "statements dropped": [int(res["statements_dropped"].sum())],
    "retries": [int((res["research_passes"] - 1).sum())],
}).T.rename(columns={0: "value"})
print("models that answered:", models)
print(f"cost per company: median ${res['cost_usd'].median():.3f}, "
      f"with a model summary ${res.loc[res['summary'], 'cost_usd'].median():.3f}, "
      f"screened only ${res.loc[res['brief'] & ~res['summary'], 'cost_usd'].median():.3f}, "
      f"existing customers ${res.loc[res['existing_customer'], 'cost_usd'].median():.3f}")
summary

**100 companies cost about $3.11 in Claude calls and took about 15 minutes. Most of that time was spent waiting
on Companies House.**

- **Time.** Fetching the records took 638 s: 407 API calls at the 0.55 s pace, with 0 failures. Running the graph
  for all 100 took 275 s, four at a time (median 10.6 s per company). A re-run from the saved states takes about
  20 s and costs nothing.
- **105 model calls.**
  - 80 briefs.
  - 23 collateral classifications: only companies with live charges whose particulars need reading.
  - 2 policy calls: only 2 companies had a pre-2013 live charge that the rules couldn't settle. Every other
    clause, for every company, was decided in code.

  Existing customers cost about $0.004 each, because their run ends before the brief.
- **Model.** Every call was answered by `claude-opus-5`, with no fallback.
- **1 statement dropped across 80 briefs.** The model cited "decision" as a source, which isn't a fact-sheet key,
  so the EVD-02 check removed it. The sentence itself was true. The check fails on the safe side.
- **Observability.** The whole batch is one LangSmith trace (`rm-copilot run · <list>`), with each
  company's graph run inside it and every Claude call inside those, carrying its tokens and cost.
  LangSmith's total for the trace and this table agree.

## 12 · Limits of this run, and what would change them

| limit | effect | fix |
|---|---|---|
| the list is a 100-company **test sample** with 25 known converters | shares (e.g. existing customers) describe the sample, not a live week | run on a real `5_score.ipynb` output |
| records are **today's**, the list was scored as of its own date | a company can change between scoring and screening | fine for a weekly run; a backtest would need records cut to the scoring date |
| `get_filing_history` keeps 60 items | CON-01/03/05 can't be decided for long histories | a date-bounded filing fetch in `mcp_ch/tools.py` |
| only the top `BRIEF_TOP_N` PROCEED/REFER leads get a model-written summary | the rest have the decision, conditions and clause table, but no plain-English summary | raise `BRIEF_TOP_N` (about $0.04 per summary) |
| the brief's faithfulness isn't scored | a sentence can say more than its source (notebook 04 §13) | an LLM-as-judge evaluation over these briefs |
| SEC-07 counts fully-satisfied charges | over-referral, and a conflict with SEC-03 on the same company (7e) | a policy-owner decision (tracked as the `xfail` test) |
| PROCEED can't judge affordability for most SMEs | CON-04 means earnings aren't public; PROCEED = no adverse indicator, not "can repay" | management accounts are collected after the approach (the CON-04 condition) |